# Bike Rental Demand Analysis

## Project Overview
Analyze bike-sharing data to understand how weather, season, holidays, and working days affect rental demand.

## Learning Objectives
- Analyze hourly and daily bike rental patterns
- Quantify the impact of weather, temperature, and season on demand
- Compare working day vs weekend/holiday usage
- Identify casual vs registered user behavior differences

## Problem Statement
Bike-sharing operators need to understand demand drivers to optimize fleet rebalancing, plan maintenance windows, and scale capacity for peak periods.

## Why This Project Matters
Bike-sharing is a growing urban transportation mode. Data-driven demand understanding improves service reliability, reduces idle bikes, and supports sustainability goals.

## Dataset Overview
Synthetic hourly bike rental dataset: 2 years (~17,520 hours) with weather conditions, temperature, humidity, wind speed, and rental counts split by casual and registered users.

## Dataset Source and License Notes
- Synthetic data inspired by the Capital Bikeshare / UCI Bike Sharing dataset
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_hours = 365 * 2 * 24
dates = pd.date_range('2022-01-01', periods=n_hours, freq='h')
hour = dates.hour
month = dates.month
dow = dates.dayofweek

# Weather
temp = np.clip(15 + 12 * np.sin(2 * np.pi * (dates.dayofyear - 80) / 365.25)
               + 4 * np.sin(2 * np.pi * (hour - 14) / 24)
               + np.random.normal(0, 3, n_hours), -5, 40).round(1)
humidity = np.clip(60 - 10 * np.sin(2 * np.pi * (hour - 14) / 24) + np.random.normal(0, 10, n_hours), 15, 100).round(0)
windspeed = np.clip(np.random.exponential(12, n_hours), 0, 50).round(1)
weather = np.random.choice(['Clear', 'Cloudy', 'Light Rain', 'Heavy Rain'], n_hours,
                             p=[0.45, 0.30, 0.18, 0.07])

# Season
season_map = {1:'Winter', 2:'Winter', 3:'Spring', 4:'Spring', 5:'Spring', 6:'Summer',
              7:'Summer', 8:'Summer', 9:'Fall', 10:'Fall', 11:'Fall', 12:'Winter'}
season = np.array([season_map[m] for m in month])
is_holiday = np.random.choice([0, 1], n_hours, p=[0.97, 0.03])
is_workday = np.where((dow < 5) & (is_holiday == 0), 1, 0)

# Demand model
hour_profile = np.array([0.05,0.03,0.02,0.02,0.03,0.08,0.20,0.55,0.70,0.50,0.40,0.45,
                          0.50,0.48,0.45,0.50,0.60,0.75,0.65,0.45,0.30,0.20,0.12,0.07])
base = np.array([hour_profile[h] for h in hour]) * 600

# Temperature effect: bell curve peaking ~22-25C
temp_effect = np.exp(-((temp - 23) ** 2) / (2 * 8**2))
# Weather penalty
weather_mult = np.array([{'Clear': 1.0, 'Cloudy': 0.85, 'Light Rain': 0.5, 'Heavy Rain': 0.15}[w] for w in weather])
# Season
season_mult = np.array([{'Spring': 1.1, 'Summer': 1.2, 'Fall': 1.0, 'Winter': 0.5}[s] for s in season])
# Working day: commute peaks; weekend: midday leisure
workday_mult = np.where(is_workday == 1, 1.0, 0.7)

total_demand = base * temp_effect * weather_mult * season_mult * workday_mult
total_demand = np.clip(total_demand + np.random.normal(0, 20, n_hours), 0, None).round(0).astype(int)

# Split casual/registered
casual_pct = np.where(is_workday == 1, 0.2, 0.45)
casual_pct = np.clip(casual_pct + np.random.normal(0, 0.05, n_hours), 0.05, 0.7)
casual = (total_demand * casual_pct).astype(int)
registered = total_demand - casual

df = pd.DataFrame({
    'Datetime': dates, 'Season': season, 'Hour': hour, 'IsHoliday': is_holiday,
    'IsWorkday': is_workday, 'Weather': weather,
    'Temperature': temp, 'Humidity': humidity, 'Windspeed': windspeed,
    'Casual': casual, 'Registered': registered, 'TotalRentals': total_demand
})
df = df.set_index('Datetime')
df['Month'] = df.index.month
df['DayOfWeek'] = df.index.day_name()

print(f'Dataset: {df.shape}')
print(f'Total rentals range: {df["TotalRentals"].min()} — {df["TotalRentals"].max()}')
print(f'Daily avg: {df["TotalRentals"].resample("D").sum().mean():,.0f}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Weather distribution:\n{pd.Series(weather).value_counts(normalize=True).round(3)}')

## Hourly Demand Profile

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
workday_profile = df[df['IsWorkday'] == 1].groupby('Hour')['TotalRentals'].mean()
weekend_profile = df[df['IsWorkday'] == 0].groupby('Hour')['TotalRentals'].mean()
axes[0].plot(workday_profile.index, workday_profile.values, marker='o', label='Workday', color='steelblue')
axes[0].plot(weekend_profile.index, weekend_profile.values, marker='s', label='Weekend/Holiday', color='coral')
axes[0].set_title('Hourly Rental Profile')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Avg Rentals')
axes[0].legend()

# Casual vs registered on workdays
cas_profile = df[df['IsWorkday'] == 1].groupby('Hour')['Casual'].mean()
reg_profile = df[df['IsWorkday'] == 1].groupby('Hour')['Registered'].mean()
axes[1].plot(cas_profile.index, cas_profile.values, marker='o', label='Casual', color='orange')
axes[1].plot(reg_profile.index, reg_profile.values, marker='s', label='Registered', color='teal')
axes[1].set_title('Casual vs Registered (Workdays)')
axes[1].set_xlabel('Hour')
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'hourly_profile.png'), dpi=100, bbox_inches='tight')
plt.show()

## Seasonal Demand

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
daily = df.resample('D')['TotalRentals'].sum()
daily.plot(ax=ax, alpha=0.4, linewidth=0.5, label='Daily')
daily.rolling(30).mean().plot(ax=ax, color='red', linewidth=2, label='30-day MA')
ax.set_title('Daily Total Rentals')
ax.set_ylabel('Rentals')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'seasonal_demand.png'), dpi=100, bbox_inches='tight')
plt.show()

## Weather Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
weather_order = ['Clear', 'Cloudy', 'Light Rain', 'Heavy Rain']
df.groupby('Weather')['TotalRentals'].mean().reindex(weather_order).plot.bar(
    ax=axes[0], edgecolor='black', color=['gold', 'gray', 'skyblue', 'navy'])
axes[0].set_title('Avg Hourly Rentals by Weather')
axes[0].tick_params(axis='x', rotation=45)

# Temperature vs demand
ax = axes[1]
temp_bins = pd.cut(df['Temperature'], bins=15)
binned = df.groupby(temp_bins, observed=True)['TotalRentals'].mean()
bin_centers = [interval.mid for interval in binned.index]
ax.plot(bin_centers, binned.values, marker='o', color='coral')
ax.set_title('Avg Rentals by Temperature')
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Avg Rentals')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'weather_impact.png'), dpi=100, bbox_inches='tight')
plt.show()

## Season and Month Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
season_order = ['Spring', 'Summer', 'Fall', 'Winter']
df.groupby('Season')['TotalRentals'].mean().reindex(season_order).plot.bar(
    ax=axes[0], edgecolor='black', color=['green', 'gold', 'orange', 'lightblue'])
axes[0].set_title('Avg Hourly Rentals by Season')
axes[0].tick_params(axis='x', rotation=0)

df.groupby('Month')['TotalRentals'].mean().plot.bar(ax=axes[1], edgecolor='black', color='teal')
axes[1].set_title('Avg Hourly Rentals by Month')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'season_month.png'), dpi=100, bbox_inches='tight')
plt.show()

## Correlation Analysis

In [ ]:
corr_cols = ['Temperature', 'Humidity', 'Windspeed', 'IsWorkday', 'Casual', 'Registered', 'TotalRentals']
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Feature Correlations')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'correlations.png'), dpi=100, bbox_inches='tight')
plt.show()

## Casual vs Registered User Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
monthly_users = df.resample('ME')[['Casual', 'Registered']].sum()
monthly_users.plot.bar(ax=ax, stacked=True, edgecolor='black')
ax.set_title('Monthly Rentals: Casual vs Registered')
ax.set_xticklabels([d.strftime('%Y-%m') for d in monthly_users.index], rotation=45, fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'casual_vs_registered.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'Overall casual %: {df["Casual"].sum() / df["TotalRentals"].sum():.1%}')

## Interpretation and Business Insight
- **Workday demand** shows classic commute peaks (8am, 5-6pm); weekends show midday leisure bump
- **Registered users** dominate workday commute hours; casual users peak on weekends and midday
- **Temperature** has a bell-curve effect — demand peaks at 20-25°C and drops in extreme cold/heat
- **Rain** dramatically reduces demand — heavy rain cuts usage by ~85%
- **Summer** is the peak season; winter demand drops ~50%
- **Fleet rebalancing** should focus on commute routes on workdays and leisure routes on weekends
- **Maintenance windows** are best scheduled during winter nights and heavy rain days

## Limitations
- Synthetic data with simplified demand model
- No station-level granularity
- No trip duration or distance data
- No rebalancing or availability data
- No membership pricing or promotion effects

## How to Improve This Project
- Add station-level demand and rebalancing analysis
- Build demand forecasting models (hourly, daily)
- Include trip origin-destination flow analysis
- Add real-time availability monitoring
- Integrate with city events and transit data

## Production Considerations
- Real-time demand dashboards per station
- Hourly demand forecasting for rebalancing trucks
- Weather-triggered capacity adjustments
- Dynamic pricing during peak demand

## Common Mistakes
- Averaging demand across all hours without separating workday/weekend profiles
- Ignoring weather interactions (e.g., warm + rainy ≠ warm + clear)
- Using total demand when casual and registered users have different patterns
- Planning capacity based on average instead of peak demand

## Mini Challenge / Exercises
1. What hour has the largest gap between workday and weekend demand?
2. Calculate the demand reduction (%) from Clear to Light Rain weather.
3. At what temperature does demand peak?
4. If the bike fleet grows by 30%, which hours and seasons would absorb the extra capacity?

## Final Summary / Key Takeaways
- Bike rental demand is driven by commute patterns (workdays) and leisure (weekends)
- Temperature and weather are the dominant external demand drivers
- Casual and registered users have fundamentally different usage patterns
- Seasonal variation is strong — winter capacity can be reduced
- Understanding demand components enables smarter rebalancing, pricing, and expansion decisions